# <font color="steelblue">Variedades de dátil</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

> **Guión de trabajo para el estudiante.** Este cuaderno es el **enunciado** de un proyecto de **aprendizaje no supervisado** que debéis **desarrollar vosotros**: objetivos, fases, tareas obligatorias, andamiaje mínimo de código (marcas `# TODO`) y criterios de evaluación. **No es una solución.**

In [ ]:
# @title Cargar configuración del cuaderno
!pip install gdown
!pip install --upgrade kagglehub
!pip install lightgbm xgboost
!pip install ucimlrepo

# Análisis numérico
import numpy as np
import pandas as pd
import math, random
import warnings
warnings.filterwarnings('ignore')

# Gráficos
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_theme(style='whitegrid')
%config InlineBackend.figure_format = 'retina'

import os, zipfile, gdown, kagglehub

# Funciones del curso
from urllib.request import urlretrieve
for fichero in ['preprocesar.py', 'evaluar_clasificadores.py', 'pca_funciones.py', 'auto_ML.py']:
    urlretrieve(f'https://raw.githubusercontent.com/ia4legos/MachineLearning/refs/heads/main/{fichero}', fichero)
from preprocesar import *
from evaluar_clasificadores import *
from pca_funciones import *
from auto_ML import *

from sklearn.decomposition import PCA, KernelPCA, IncrementalPCA, FastICA
from sklearn.cluster import KMeans, MiniBatchKMeans, BisectingKMeans
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering, FeatureAgglomeration
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS, Isomap, LocallyLinearEmbedding, TSNE, SpectralEmbedding

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, davies_bouldin_score,
                             accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             adjusted_rand_score)

RNG = 42

## Cargar funciones accesorias

############# Funciones MDS ##############
def stress1(D_orig, embedding):
    """Stress-1 de Kruskal entre las distancias originales y las del embedding.
    Portable entre versiones de scikit-learn (no depende del atributo stress_)."""
    D_proj = pairwise_distances(embedding)
    iu = np.triu_indices_from(D_orig, k=1)          # pares (i<j), sin diagonal
    num = np.sum((D_orig[iu] - D_proj[iu])**2)
    den = np.sum(D_proj[iu]**2)
    return np.sqrt(num / den)

def ajustar_mds(D, n_components=2, metrico=True, n_init=4, random_state=RNG):
    """Ajusta un MDS (métrico o no) sobre una matriz de distancias precalculada
    y devuelve el embedding. Usa 'metric'/'dissimilarity' por compatibilidad amplia."""
    modelo = MDS(n_components=n_components, dissimilarity='precomputed',
                 metric=metrico, n_init=n_init, random_state=random_state)
    return modelo.fit_transform(D)

def curva_stress(D, metrico=True, max_dim=10, n_init=4):
    """Calcula el Stress-1 para soluciones de 1 a max_dim dimensiones."""
    dims = range(1, max_dim + 1)
    valores = [stress1(D, ajustar_mds(D, d, metrico, n_init)) for d in dims]
    return list(dims), valores

def grafico_codo(dims, valores, titulo=''):
    """Curva de Stress-1 vs nº de dimensiones, con las bandas de calidad de Kruskal."""
    plt.figure(figsize=(8, 5))
    plt.plot(dims, valores, 'o-')
    for y, txt, col in [(0.05, 'bueno', 'green'), (0.10, 'aceptable', 'orange'), (0.20, 'pobre', 'red')]:
        plt.axhline(y, ls='--', lw=1, color=col, alpha=.6)
        plt.text(dims[-1], y, f'  {txt}', color=col, va='center')
    plt.xticks(dims); plt.xlabel('Nº de dimensiones'); plt.ylabel('Stress-1')
    plt.title(titulo); plt.show()

def grafico_distancias(D_orig, embedding, titulo='Distancias proyectadas vs originales'):
    """Gráfico distancia-distancia: nube sobre la diagonal => buen ajuste."""
    D_proj = pairwise_distances(embedding)
    m = np.max(D_orig)
    plt.figure(figsize=(8, 6))
    plt.scatter(D_orig, D_proj, s=8, alpha=.3, label='pares de puntos')
    plt.plot([0, m], [0, m], 'k-', lw=2, label='ajuste perfecto (y = x)')
    plt.xlabel('Distancia original'); plt.ylabel('Distancia proyectada')
    plt.title(titulo); plt.legend(); plt.show()

def nearest_neighbors(X, k):
    """Matriz (n, k) con los índices de los k vecinos más próximos de cada punto
    (distancia euclídea), excluyendo el propio punto. Devuelve enteros."""
    D = pairwise_distances(X)
    # ordena cada fila por distancia; la columna 0 es el propio punto (dist. 0)
    return np.argsort(D, axis=1)[:, 1:k+1]

def dibujar_grafo(X, knn, ax=None, **kw):
    """Dibuja las aristas punto–vecino sobre un eje 2D."""
    ax = ax or plt.gca()
    for i, vecinos in enumerate(knn):
        for j in vecinos:
            ax.plot(X[[i, j], 0], X[[i, j], 1], c='gray', lw=0.6, zorder=1, **kw)

########### Cluster ##############
def plot_dendrogram(model, **kwargs):
    """Dibuja el dendrograma a partir de un AgglomerativeClustering ya ajustado
    con compute_distances=True (o distance_threshold). Acepta los mismos kwargs
    que scipy.cluster.hierarchy.dendrogram (truncate_mode, p, color_threshold, ax...)."""
    counts = np.zeros(model.children_.shape[0])     # nº de muestras bajo cada fusión
    n = len(model.labels_)
    for i, merge in enumerate(model.children_):
        c = 0
        for child in merge:
            c += 1 if child < n else counts[child - n]   # hoja (1) o sub-cluster (su recuento)
        counts[i] = c
    Z = np.column_stack([model.children_, model.distances_, counts]).astype(float)
    return dendrogram(Z, **kwargs)

def plot_scree(X, metric, linkage_m, kmin, kmax, **kwargs):
    """Índice de silueta para soluciones jerárquicas de kmin a kmax grupos."""
    ks = range(kmin, kmax)
    sil = [silhouette_score(X, AgglomerativeClustering(n_clusters=k, metric=metric,
                                                       linkage=linkage_m).fit_predict(X)) for k in ks]
    plt.plot(list(ks), sil, marker='o', **kwargs)
    plt.xlabel('número de grupos'); plt.ylabel('silueta media'); plt.xticks(list(ks))

from matplotlib.patches import Ellipse
def draw_ellipse(position, cov2x2, ax=None, **kwargs):
    """Dibuja la elipse (a 1, 2 y 3 sigmas) dada una covarianza 2x2."""
    ax = ax or plt.gca()
    U, s, Vt = np.linalg.svd(cov2x2)
    angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
    width, height = 2 * np.sqrt(s)
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height,
                             angle=angle, **kwargs))   # 'angle' es keyword en matplotlib actual

def _cov_2x2(gmm, k):
    """Reconstruye la covarianza 2x2 de la componente k según covariance_type."""
    ct = gmm.covariance_type
    if ct == 'full':      return gmm.covariances_[k]
    if ct == 'tied':      return gmm.covariances_
    if ct == 'diag':      return np.diag(gmm.covariances_[k])
    return np.eye(2) * gmm.covariances_[k]            # 'spherical'

def plot_gmm(gmm, X, label=True, ax=None):
    """Ajusta el GMM y dibuja puntos coloreados por componente + sus elipses
    (válido para los cuatro covariance_type)."""
    ax = ax or plt.gca()
    labels = gmm.fit(X).predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=labels if label else None,
               s=20, cmap='viridis', zorder=2)
    ax.set_aspect('equal')
    w_factor = 0.3 / gmm.weights_.max()
    for k, (pos, w) in enumerate(zip(gmm.means_, gmm.weights_)):
        draw_ellipse(pos, _cov_2x2(gmm, k), ax=ax, alpha=w * w_factor, color='steelblue')

def draw_ellipsoid(ax, mean, cov, nsig=2, **kw):
    # dibuja el elipsoide a 'nsig' sigmas de una gaussiana 3D (media + cov 3x3)
    vals, vecs = np.linalg.eigh(cov)                 # autovalores/autovectores
    u = np.linspace(0, 2*np.pi, 24); v = np.linspace(0, np.pi, 16)
    esfera = np.array([np.outer(np.cos(u), np.sin(v)),
                       np.outer(np.sin(u), np.sin(v)),
                       np.outer(np.ones_like(u), np.cos(v))])
    r = nsig * np.sqrt(vals)                          # longitudes de los semiejes
    pts = np.einsum('ij,jkl->ikl', vecs, esfera * r[:, None, None])
    ax.plot_surface(pts[0]+mean[0], pts[1]+mean[1], pts[2]+mean[2], **kw)

## <font color="steelblue">Objetivos del proyecto</font>

Cada dátil está descrito por **34 características** extraídas por visión artificial y pertenece a una de **7 variedades**. Aquí **ignoramos la variedad para agrupar** (aprendizaje no supervisado) y luego **comparamos la solución de cluster con la clase real**. El objetivo es doble:

1. **Descubrir la estructura** de los datos sin usar la etiqueta: reducir dimensión (PCA/MDS) y **agrupar** con varios algoritmos.
2. **Validar externamente** la solución: ¿los grupos hallados **se corresponden** con las variedades reales? Lo mediremos con **matriz de contingencia, ARI/NMI, mapeo óptimo cluster→variedad**.



## <font color="steelblue">El conjunto de datos</font>

**898 dátiles** (Koklu et al., 2021), **34 características numéricas** + `Class` (**7 variedades**, nominal). **Sin valores perdidos.** Las características se agrupan en cuatro familias:

* **Morfología (tamaño):** `AREA`, `PERIMETER`, `MAJOR_AXIS_LENGTH`, `MINOR_AXIS_LENGTH`, `EQUIVALENT_DIAMETER`, `CONVEX_AREA`.
* **Forma:** `ECCENTRICITY`, `SOLIDITY`, `EXTENT`, `ROUNDNESS`, `ASPECT_RATIO`, `COMPACTNESS`, `SHAPEFACTOR_1..4`.
* **Color (estadísticos RGB):** `MeanR/G/B`, `StdDevR/G/B`, `SkewR/G/B`, `KurtosisR/G/B`, `EntropyR/G/B`.
* **Textura (wavelet Daubechies-4):** `ALLdaub4R/G/B`.
* **Clase `Class`:** `Barhee`, `Deglet Nour`, `Sukkary`, `Rotab Mozafati`, `Ruthana`, `Safawi`, `Sagai`. **Desequilibrio moderado** (de 65 a 204).

> **Claves:** (1) escalas **muy** dispares → **escalado** imprescindible; (2) **colinealidad fuerte** entre características de ingeniería (muchas son funciones de otras); (3) la clase **se reserva** y **no** entra en el clustering: solo se usa para la **comparación** final.

## <font color="steelblue">Reglas del juego</font>

1. **Reserva `Class`**: el clustering se hace **solo** con las 34 características. La etiqueta se usa **únicamente** en la fase de comparación.
2. **Escala** antes de PCA/clustering (las escalas van de [0,1] a miles de píxeles).
3. **Trata la redundancia/colinealidad** (selección/PCA) y coméntala.
4. **Combina modelos:** reduce dimensión + agrupa; compara **varios** algoritmos.
5. **Elige el nº de grupos por criterios internos** y **discútelo frente a las 7 variedades conocidas** (¿coincide el `k` óptimo con 7?).
6. **Comparación NOMINAL:** ARI/NMI + **mapeo óptimo cluster→variedad** + `classification_report`. **No** uses métricas ordinales (las variedades no tienen orden).
7. **Caracteriza** los grupos y **reproducibilidad** con `random_state`.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
path = kagglehub.dataset_download("muratkokludataset/date-fruit-datasets")
archivos = os.listdir(path); print("Archivos:", archivos)
archivo_csv  = [f for f in archivos if f.endswith('.csv')]
archivo_xlsx = [f for f in archivos if f.endswith('.xlsx')]
if archivo_csv:
    df = pd.read_csv(os.path.join(path, archivo_csv[0]))
elif archivo_xlsx:
    df = pd.read_excel(os.path.join(path, archivo_xlsx[0]))
else:
    raise FileNotFoundError("No se encontró CSV ni XLSX.")
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA</font>

**Tareas a desarrollar**
1. **Clase.** Distribución de `Class` (7 variedades): constata el **desequilibrio moderado**. (La reservaremos para el final.)
2. **Grupos de características.** Construye las listas de las cuatro familias (morfología, forma, color, textura).
3. **Escalas.** Compara rangos (`AREA` vs `ECCENTRICITY`…): justifica el **escalado**.
4. **Colinealidad.** Matriz de correlación: localiza bloques muy correlacionados.
5. **Anticipo visual.** Proyecta con **PCA**/**t-SNE** coloreando por la **clase real**: ¿se separan ya las variedades? (esto adelanta la comparación final).
6. **Conclusión:** 3–4 hallazgos.

# <font color="steelblue">Fase 2 — Preprocesado</font>

**Tareas a desarrollar**
1. **Separa** `y = Class` (resérvala) y `X` = las **34 características**.
2. **Escala** `X` (StandardScaler). Conserva `X` original para interpretar.
3. **(Opcional) Redundancia:** valora PCA o selección para mitigar la colinealidad y compara.
4. Guarda las **listas de grupos** (Fase 1) para el análisis por familias (Fase 4/7).

> **A responder:** ¿por qué el clustering **no** debe ver `Class`? ¿qué validez tendría la comparación si la hubieras usado?

# <font color="steelblue">Fase 3 — Reducción de dimensión</font>

**Tareas a desarrollar**
1. **PCA** sobre `X` escalado: **scree plot** e **interpretación de loadings** (¿qué mide PC1? ¿tamaño? ¿color?).
2. **Visualización.** Proyecta en PC1–PC2 y con **MDS**/**t-SNE**, coloreando por la **clase real**: ¿la estructura no supervisada separa las variedades?
3. **Conclusión:** dimensionalidad efectiva y qué capturan las componentes.

# <font color="steelblue">Fase 4 — Agrupación (clustering) y combinación de modelos</font>

**Tareas a desarrollar**
1. Aplica y compara **≥3** algoritmos: **K-Means**, **jerárquico** (+ dendrograma) y **DBSCAN** y/o **GMM**.
2. **Combina con la reducción:** agrupa sobre el **espacio PCA** y compáralo con `X` escalado.
3. **(Recomendado) Por grupos de características:** repite el clustering usando **solo color**, **solo forma**, **solo morfología**, **solo textura** y **todas**: ¿qué familia recupera mejor las variedades? (lo cuantificarás en la Fase 6).
4. **Concordancia** entre soluciones (Adjusted Rand Index).

> Explora `k` en la Fase 5; para comparar con 7 variedades puedes mirar `k=7`, sin forzar.

# <font color="steelblue">Fase 5 — Validación interna y número de grupos</font>

**Tareas a desarrollar**
1. **Número de grupos:** codo (inercia) y **silueta** variando `k`; dendrograma para el jerárquico.
2. **Índices internos:** silueta, **Davies–Bouldin**, **Calinski–Harabasz**.
3. **Estabilidad** ante semillas/submuestras.
4. **Discusión clave:** ¿el `k` que sugieren los criterios internos **coincide con las 7 variedades**? Si no, ¿por qué (variedades solapadas, sub/superagrupación)?

# <font color="steelblue">Fase 6 — Comparación con la clase real</font>

La fase **distintiva**: contrastar la **solución de cluster** con la **variedad real** `Class`. Como la clase es **nominal** (sin orden), se usan métricas **nominales**.

**Tareas a desarrollar**
1. **Matriz de contingencia** grupos × variedad (`pd.crosstab`): ¿cada cluster se asocia a una variedad?
2. **Índices externos:** **Adjusted Rand Index** y **NMI** entre la partición y la clase.
3. **Mapeo óptimo cluster→variedad.** Usa el **algoritmo húngaro** (`scipy.optimize.linear_sum_assignment`) sobre la contingencia para la asignación que **maximiza los aciertos**, calcula la **"clustering accuracy"** y genera un **`classification_report`** frente a `Class`. (Alternativa simple: cluster→clase mayoritaria.)
4. **¿Qué se confunde?** Identifica variedades que el clustering **fusiona** o **parte**, y relaciónalo con el solapamiento visto en PCA/t-SNE.
5. **Por grupos de características (si hiciste la Fase 4.3):** compara ARI/accuracy clusterizando con color/forma/morfología/textura/todas: **¿qué familia recupera mejor las variedades?**
6. **Interpretación crítica:** un ARI alto indica que la estructura **natural** coincide con las variedades; uno bajo, que las variedades **no** son los grupos más salientes (otras fuentes de variación dominan).

# <font color="steelblue">Fase 7 — Caracterización de los grupos</font>

**Tareas a desarrollar**
1. **Perfiles.** Media de cada característica (valores **originales**) por grupo → **mapa de calor**.
2. **Rasgos distintivos.** Qué define a cada grupo (z-score): ¿tamaño? ¿color? ¿forma?
3. **¿Grupos = variedades?** Cruza los perfiles con la variedad dominante de cada cluster (de la Fase 6) y nombra los grupos en términos **interpretables** (no solo el nombre de la variedad).
4. **Lectura aplicada.** Utilidad para clasificación/cribado automático en una línea de producción.

# <font color="steelblue">Fase 8 — Aplicación interactiva (widget en Colab)</font>

Aplicación **dentro del cuaderno** con **`ipywidgets`** (no despliegue). Implementa **al menos una**:

1. **Explorador de grupos:** desplegable de grupo → perfil medio (barras/radar), tamaño, **variedad dominante** y pureza.
2. **Asignador:** *sliders* con las características principales → el widget **escala → (PCA) → asigna el cluster** más cercano y muestra la **variedad probable** (según el mapeo de la Fase 6).
3. **(Opcional)** Dispersión PC1–PC2 con un selector "colorear por cluster / por variedad real" (`plotly`) para **ver la coincidencia**.

> El widget solo **usa** el modelo ya ajustado; no reentrena en cada interacción.

# <font color="steelblue">Entregables y formato</font>

* **Cuaderno** ejecutable de principio a fin, con código y **celdas de texto** que justifiquen cada decisión (escalado, nº de grupos, **interpretación de la comparación**).
* **Memoria breve:** EDA, reducción de dimensión, comparación de algoritmos, validación interna, **comparación con la clase real (contingencia + ARI/NMI + mapeo óptimo + `classification_report`)**, caracterización y la aplicación.
* **Widget(s)** funcionando dentro del cuaderno.
* **Reproducibilidad:** `random_state` fijado y comentario de estabilidad.

## <font color="steelblue">Rúbrica de evaluación (100 pts)</font>

| Fase | Qué se valora | Puntos |
|---|---|---:|
| 1. EDA | 7 clases, grupos de características, escalas, color por clase | 12 |
| 2. Preprocesado | **Reservar `Class`**, escalado, redundancia | 12 |
| 3. Reducción de dimensión | PCA (interpretación) + MDS/t-SNE coloreado por clase | 14 |
| 4. Clustering y combinación | ≥3 algoritmos, PCA vs original, (por grupos de características) | 14 |
| 5. Validación interna | Codo/silueta, índices, estabilidad, **¿k==7?** | 12 |
| 6. **Comparación con la clase real** | Contingencia + ARI/NMI + **mapeo óptimo (húngaro)** + `classification_report`; nominal (sin QWK/MAE) | 20 |
| 7. Caracterización | Perfiles, rasgos, **grupos↔variedades**, lectura aplicada | 10 |
| 8. Aplicación (widget) | `ipywidgets` funcional | 6 |
| — Extra | Ablación por grupos de características, UMAP, consenso de clustering, comparación con la solución supervisada | +10 |

> **Penalizaciones:** **usar `Class` en el clustering** (invalida la comparación), **no escalar**, evaluar con **métricas ordinales** (la clase es nominal), o limitarse a soltar números **sin interpretar** qué variedades se confunden.

# <font color="steelblue">Pistas y errores típicos</font>

* **Reserva la clase.** Si `Class` entra en el clustering, la comparación posterior no significa nada.
* **Escala.** Con escalas de [0,1] a miles de píxeles, kNN/distancias quedan dominadas por el tamaño; escala siempre.
* **k interno vs 7.** Es habitual que el mejor `k` por silueta **no** sea 7 (variedades solapadas o subgrupos): eso es un **hallazgo**, no un error.
* **Mapeo óptimo.** El algoritmo **húngaro** asigna cluster→variedad maximizando aciertos; evita el sesgo del "mayoritario" cuando varios clusters comparten variedad dominante.
* **Nominal, no ordinal.** Variedades sin orden → ARI/NMI/macro-F1, **no** QWK/MAE (al contrario que el proyecto de Amazon, con etiqueta ordinal).
* **Widget, no despliegue:** `ipywidgets` dentro de Colab.

# <font color="steelblue">Referencias</font>

* Koklu, M., Kursun, R., Taspinar, Y. S. & Cinar, I. (2021). *Classification of Date Fruits into Genetic Varieties Using Image Analysis*. Mathematical Problems in Engineering.
* *Date Fruit Classification with Machine Learning and Explainable AI (LIME/SHAP)* (2023).
* Cuadernos del curso: *Componentes principales y variantes*, *Escalas multidimensionales (MDS)*, *Métodos de manifold*, *Clustering (K-Means, jerárquico, DBSCAN, mixturas gaussianas)*.
